<!--
---
PURPOSE: Discover videos and align frame times to NWB timebase.
REQUIRES:
  - outputs/reports/session_inventory.parquet
PRODUCES:
  - outputs/video/video_assets.parquet
  - outputs/video/frame_times.parquet
WHAT_NEXT: notebooks/06_Pose_Estimation_Setup_SLEAP_or_DLC.ipynb
---
-->

# 05 Video I/O and Frame Timebase

**Purpose**
Discover videos and align frame times to NWB timebase.

**Requires**
- `outputs/reports/session_inventory.parquet`

**Produces**
- `outputs/video/video_assets.parquet`
- `outputs/video/frame_times.parquet`

**What to run next**
- `notebooks/06_Pose_Estimation_Setup_SLEAP_or_DLC.ipynb`

We locate videos, compute per-frame timestamps, and validate alignment.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()      
ROOT = ROOT.parent              
SRC  = ROOT / "src"           

# put repo + src on sys.path
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


#print("ROOT:", ROOT)
#print("SRC :", SRC, "| exists:", SRC.exists())
#print("sys.path[0:3]:", sys.path[:3])

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [ ]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = ROOT / "notebooks" / "05_Video_IO_and_Frame_Timebase.ipynb"
header  = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))

if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

## Step 1: Build video manifest and frame times
This step enforces the timebase alignment tiers and produces QC flags.

In [ ]:
from io_sessions import load_sessions_csv, get_session_bundle
from viz import plot_video_alignment

sessions = load_sessions_csv()
SESSION_IDS = sessions.session_id.tolist()[:1]

for session_id in SESSION_IDS:
    bundle = get_session_bundle(session_id, sessions)
    assets = bundle.load_video_assets()
    frame_times = bundle.load_frame_times()
    if assets is not None and not assets.empty:
        print(assets[["camera", "source", "n_frames", "qc_flags"]])
    else:
        print("No video assets available.")
    if frame_times is not None and not frame_times.empty:
        for camera in sorted(frame_times["camera"].unique()):
            ft_cam = frame_times[frame_times["camera"] == camera]
            print(f"{session_id=} {camera=} frames={len(ft_cam)}")
            plot_video_alignment(ft_cam)
